# Asynchronous Federated Learning — Independent Virtual Clients (K=5)

This notebook modifies the original **independent-client discrete-event simulation** into a **buffered asynchronous FL baseline**. The server aggregates **exactly 5 arriving updates** at a time. Each update is a complete locally trained model, and aggregation uses **plain sample-weighted FedAvg of those five models**. There is **no staleness correction, weighting, filtering, rebasing, or mixing with the latest server model**.

Clients run on independent **virtual timelines**. Actual PyTorch batches execute when their virtual training events are handled, so CPU/GPU wall time is **not** true concurrent training time. Virtual seconds are determined by the configured client speeds and download/upload latencies. This distinction is critical when interpreting throughput and overall time.

For comparison against the previous synchronous 5-round / 10-client baseline (**50 updates**), this run completes **50 client updates**: exactly 10 buffers of 5 updates. Compare using **updates received**, **virtual time**, and **accuracy-versus-time**, not only aggregation count. The original synchronous run has 5 global aggregations; this asynchronous run has 10.


## 1. Same model, dataset and simulation configuration as the synchronous notebook

In [ ]:
import math
import copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
NUM_CLIENTS = 10
NUM_GLOBAL_ROUNDS = 5
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
LEARNING_RATE = 0.03
MOMENTUM = 0.9
TRAIN_SAMPLES = 12_000
TEST_SAMPLES = 2_000
TIME_QUANTUM = 1.0
CLIENT_SPEEDS = [1.0, 1.5, 0.5, 1.0, 2.0, 0.35, 1.2, 0.75, 1.0, 0.6]
DOWNLOAD_LATENCY = [0, 0, 2, 0, 1, 5, 0, 3, 0, 4]
UPLOAD_LATENCY = [1, 1, 4, 1, 1, 6, 2, 3, 1, 5]
PARTITION_MODE = 'dirichlet'  # or 'iid'
DIRICHLET_ALPHA = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Running on:', DEVICE)
# Concise console logging: queue preview only at the start of each round.
QUEUE_PREVIEW_SIZE = 6

# New asynchronous-only settings; all original model/data/client parameters remain unchanged.
AGGREGATION_BUFFER_SIZE = 5
TARGET_UPDATES = (NUM_CLIENTS * NUM_GLOBAL_ROUNDS // AGGREGATION_BUFFER_SIZE) * AGGREGATION_BUFFER_SIZE
# No correction or enhancement: direct sample-weighted FedAvg of the 5 received states.
ASYNC_METRICS_FILE = 'async_staleness_unweighted_k5_metrics.csv'
BASELINE_METRICS_FILE = 'round_metrics.csv'  # Optional file exported by the synchronous notebook.


## 2. Original MNIST selection and client partitioning

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset_full = datasets.MNIST('./data', train=False, download=True, transform=transform)
rng = np.random.default_rng(SEED)
train_indices = rng.choice(len(train_dataset_full), size=TRAIN_SAMPLES, replace=False)
test_indices = rng.choice(len(test_dataset_full), size=TEST_SAMPLES, replace=False)
test_dataset = Subset(test_dataset_full, test_indices.tolist())

def iid_partition(indices, num_clients):
    shuffled = np.array(indices).copy()
    np.random.default_rng(SEED).shuffle(shuffled)
    return [chunk.tolist() for chunk in np.array_split(shuffled, num_clients)]

def dirichlet_partition(dataset, selected_indices, num_clients, alpha):
    rng = np.random.default_rng(SEED)
    labels = np.asarray(dataset.targets)[selected_indices]
    # Retry if a client would have no samples.
    for _ in range(100):
        result = [[] for _ in range(num_clients)]
        for class_id in np.unique(labels):
            indices = np.asarray(selected_indices)[labels == class_id].copy()
            rng.shuffle(indices)
            p = rng.dirichlet(np.full(num_clients, alpha))
            chunks = np.split(indices, (np.cumsum(p)[:-1] * len(indices)).astype(int))
            for i, chunk in enumerate(chunks): result[i].extend(chunk.tolist())
        if all(result):
            for group in result: rng.shuffle(group)
            return result
    raise RuntimeError('Could not form nonempty client partitions; increase alpha.')

client_partitions = (iid_partition(train_indices, NUM_CLIENTS) if PARTITION_MODE == 'iid'
    else dirichlet_partition(train_dataset_full, train_indices, NUM_CLIENTS, DIRICHLET_ALPHA))
print('Samples per client:', [len(p) for p in client_partitions])

## 3. Original neural network, client training, FedAvg, evaluation

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

class SimulatedClient:
    def __init__(self, client_id, indices, speed, download_latency, upload_latency):
        self.client_id = client_id
        self.indices = list(indices)
        self.num_samples = len(indices)
        self.speed = speed
        self.download_latency = download_latency
        self.upload_latency = upload_latency

    def start_round(self, global_state, current_time, round_number):
        self.round_number = round_number
        self.model = SimpleNN().to(DEVICE)
        self.model.load_state_dict(copy.deepcopy(global_state))
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
        self.epoch = 0
        self.batch_cursor = 0
        self.work_credit = 0.0
        self.training_finished = False
        self.update_arrived = False
        self.finish_time = None
        self.arrival_time = None
        self.ready_time = current_time + self.download_latency
        self.total_batches_trained = 0
        self.running_loss = 0.0
        self._shuffle_epoch()

    def _shuffle_epoch(self):
        self.shuffled_indices = self.indices.copy()
        random.shuffle(self.shuffled_indices)
        self.batch_cursor = 0

    def _train_one_batch(self):
        batch_ids = self.shuffled_indices[self.batch_cursor:self.batch_cursor + BATCH_SIZE]
        images, labels = zip(*(train_dataset_full[int(i)] for i in batch_ids))
        x = torch.stack(images).to(DEVICE)
        y = torch.tensor(labels, dtype=torch.long, device=DEVICE)
        self.model.train()
        self.optimizer.zero_grad()
        loss = F.cross_entropy(self.model(x), y)
        loss.backward()
        self.optimizer.step()
        self.running_loss += loss.item()
        self.total_batches_trained += 1
        self.batch_cursor += len(batch_ids)
        if self.batch_cursor >= len(self.shuffled_indices):
            self.epoch += 1
            if self.epoch >= LOCAL_EPOCHS: return True
            self._shuffle_epoch()
        return False

    def execute_time_slice(self, current_time, quantum):
        if self.training_finished or current_time < self.ready_time:
            return 0
        self.work_credit += self.speed * quantum
        batches = 0
        while self.work_credit + 1e-9 >= 1 and not self.training_finished:
            finished = self._train_one_batch()
            self.work_credit -= 1.0
            batches += 1
            if finished:
                self.training_finished = True
                # This baseline places completion at the END of the scheduled quantum.
                self.finish_time = current_time + quantum
                self.arrival_time = self.finish_time + self.upload_latency
        return batches

    def get_model_state(self):
        return {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}

def fedavg(states, weights):
    total = sum(weights)
    return {key: sum(states[i][key] * (weights[i] / total) for i in range(len(states)))
            for key in states[0]}

@torch.no_grad()
def evaluate(model):
    model.eval()
    loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        output = model(x)
        total_loss += F.cross_entropy(output, y, reduction='sum').item()
        correct += (output.argmax(1) == y).sum().item()
        total += y.numel()
    return {'loss': total_loss / total, 'accuracy': correct / total}

**Logging:** concise initial client/version list, initial priority queue preview, and one summary **per aggregation of 5 updates**. Each summary reports arriving clients, their trained model versions, staleness, global version change, elapsed virtual time, accuracy, and throughput. Every event is retained in CSV without printing training quanta.

**How staleness arises:** A client downloads global model vN when its download finishes, then trains/uploads while other updates may trigger vN+1, vN+2, etc. When its update finally arrives, the buffer accepts the stale model **unchanged**. Plain FedAvg is applied directly to the five models in each buffer. Clients fetch whatever version exists at the time of their next download completion.


## 4. Discrete-event queue and reporting helpers

In [ ]:
import heapq
import itertools

EVENT_PRIORITY = {'DOWNLOAD_COMPLETE': 0, 'UPDATE_ARRIVED': 1, 'QUANTUM_END': 2}

class EventQueue:
    def __init__(self):
        self.heap = []
        self.counter = itertools.count()

    def push(self, when, kind, client_id=-1):
        heapq.heappush(self.heap, (float(when), EVENT_PRIORITY[kind],
                                   next(self.counter), kind, client_id))

    def pop(self):
        when, _, _, kind, client_id = heapq.heappop(self.heap)
        return when, kind, client_id

    def __bool__(self):
        return bool(self.heap)


def queue_snapshot(queue, limit=QUEUE_PREVIEW_SIZE):
    return [f't={t:.1f} {kind} C{cid:02d} (p={priority})'
            for t, priority, seq, kind, cid in sorted(queue.heap)[:limit]]


def print_queue(queue, heading='PRIORITY QUEUE'):
    preview = queue_snapshot(queue)
    print(f'  {heading} | pending={len(queue.heap)} | sorted by time, priority, insertion')
    for i, event in enumerate(preview, 1):
        print(f'    {i:02d}. {event}')
    if len(queue.heap) > len(preview):
        print(f'    ... {len(queue.heap)-len(preview)} additional events')


## 5. Continuous event-driven scheduling with aggregation on every 5 arrivals

- Start one job for each of 10 clients; independent virtual timelines determine arrivals.
- After an upload reaches the server, the client is eligible to download the **current** model for a new job. Each job trains **2 local epochs**, just as before.
- The server aggregates **as soon as the fifth update reaches it**, regardless of how stale the five client versions are.
- No waiting for slow clients, no staleness corrections, and **no synchronous round barrier**.
- Stop after **50 arrived updates / 10 aggregations** (near the original baseline's 50 client updates). Virtual elapsed time and measured execution wall time are both recorded.


In [ ]:
import time

experiment_wall_start = time.perf_counter()
clients = [SimulatedClient(i, client_partitions[i], CLIENT_SPEEDS[i],
                           DOWNLOAD_LATENCY[i], UPLOAD_LATENCY[i])
           for i in range(NUM_CLIENTS)]
global_model = SimpleNN().to(DEVICE)
global_state = copy.deepcopy(global_model.state_dict())
global_version = 0
virtual_time = 0.0

event_log, update_records, agg_metrics, queue_history = [], [], [], []
queue = EventQueue()
started_jobs = 0
arrived_updates = 0
buffer = []

# Reserve jobs when enqueueing downloads, so the run ends at exactly TARGET_UPDATES.
def schedule_download(c, now):
    global started_jobs
    if started_jobs >= TARGET_UPDATES:
        return False
    started_jobs += 1
    c.pending_job = started_jobs
    queue.push(now + c.download_latency, 'DOWNLOAD_COMPLETE', c.client_id)
    return True

for c in clients:
    schedule_download(c, 0.0)

print(f'ASYNCHRONOUS FL | buffer={AGGREGATION_BUFFER_SIZE} | target updates={TARGET_UPDATES}')
print(f'Initial server version v{global_version}; all clients initially download v0.')
print('CLIENTS (ID | speed in virtual batches/s | download | upload)')
for c in clients:
    print(f'  C{c.client_id:02d} | {c.speed:.2f} | {c.download_latency:.1f}s | {c.upload_latency:.1f}s')
print_queue(queue, 'INITIAL PRIORITY QUEUE')

# Initial test score is useful for fair convergence comparisons.
initial_eval = evaluate(global_model)
agg_metrics.append(dict(aggregation=0, global_version=0, start_time=0.0,
    end_time=0.0, duration=0.0, updates_received=0, updates_aggregated=0,
    accuracy=initial_eval['accuracy'], loss=initial_eval['loss'],
    cumulative_throughput=0.0, window_throughput=0.0,
    mean_staleness=0.0, max_staleness=0, wall_elapsed_s=time.perf_counter()-experiment_wall_start))

last_aggregation_time = 0.0
while queue and arrived_updates < TARGET_UPDATES:
    when, kind, client_id = queue.pop()
    virtual_time = when
    c = clients[client_id]

    if kind == 'DOWNLOAD_COMPLETE':
        # Model is SNAPSHOTTED at download completion, not when the job was queued.
        c.start_round(global_state, when, c.pending_job)
        c.model_version = global_version
        c.job_number = c.pending_job
        c.download_time = when
        # start_round computes ready_time=now+download_latency, so override it:
        # this event *already means* the download finished.
        c.ready_time = when
        event_log.append(dict(virtual_time=when, event=kind, client=client_id,
                              job=c.job_number, model_version=c.model_version))
        queue.push(when + TIME_QUANTUM, 'QUANTUM_END', client_id)

    elif kind == 'QUANTUM_END':
        batches = c.execute_time_slice(when - TIME_QUANTUM, TIME_QUANTUM)
        # No verbose per-batch/per-quantum printout; preserve all events in CSV.
        event_log.append(dict(virtual_time=when, event=kind, client=client_id,
                              job=c.job_number, model_version=c.model_version, batches=batches))
        if c.training_finished:
            queue.push(c.arrival_time, 'UPDATE_ARRIVED', client_id)
        else:
            queue.push(when + TIME_QUANTUM, 'QUANTUM_END', client_id)

    elif kind == 'UPDATE_ARRIVED':
        arrived_updates += 1
        staleness = global_version - c.model_version
        assert staleness >= 0
        state = c.get_model_state()
        buffer.append(dict(client=c.client_id, state=state, samples=c.num_samples,
                           trained_version=c.model_version, arrival_version=global_version,
                           staleness=staleness, arrival_time=when, job=c.job_number))
        update_records.append(dict(update=arrived_updates, client=c.client_id, job=c.job_number,
             trained_version=c.model_version, server_version_at_arrival=global_version,
             staleness=staleness, samples=c.num_samples, batches=c.total_batches_trained,
             download_time=c.download_time, training_finish_time=c.finish_time,
             arrival_time=when, train_loss=c.running_loss/max(c.total_batches_trained,1)))
        event_log.append(dict(virtual_time=when, event=kind, client=client_id,
                              job=c.job_number, model_version=c.model_version,
                              staleness=staleness))

        if len(buffer) == AGGREGATION_BUFFER_SIZE:
            old_version = global_version
            # Intentional naive baseline: aggregate full client models, even if stale.
            # No staleness scaling, no global-model blending, no delta rebasing.
            global_state = fedavg([x['state'] for x in buffer], [x['samples'] for x in buffer])
            global_model.load_state_dict(global_state)
            global_version += 1
            stats = evaluate(global_model)
            window_duration = when - last_aggregation_time
            window_throughput = AGGREGATION_BUFFER_SIZE/window_duration if window_duration > 0 else float('nan')
            cumulative_throughput = arrived_updates/when if when > 0 else float('nan')
            avg_staleness = float(np.mean([x['staleness'] for x in buffer]))
            max_staleness = max(x['staleness'] for x in buffer)
            agg_metrics.append(dict(aggregation=global_version, global_version=global_version,
                start_time=last_aggregation_time, end_time=when, duration=window_duration,
                updates_received=arrived_updates, updates_aggregated=AGGREGATION_BUFFER_SIZE,
                accuracy=stats['accuracy'], loss=stats['loss'],
                cumulative_throughput=cumulative_throughput, window_throughput=window_throughput,
                mean_staleness=avg_staleness, max_staleness=max_staleness,
                wall_elapsed_s=time.perf_counter()-experiment_wall_start))
            members = ', '.join(f'C{x["client"]:02d}(v{x["trained_version"]}, stale={x["staleness"]})' for x in buffer)
            print(f'AGG {global_version:02d} | t={when:7.1f}s | {members} | '
                  f'global v{old_version}->v{global_version} | '
                  f'acc={stats["accuracy"]*100:5.2f}% loss={stats["loss"]:.4f} | '
                  f'updates={arrived_updates} throughput={cumulative_throughput:.3f}/s')
            event_log.append(dict(virtual_time=when, event='AGGREGATE', client=-1,
                                  job=-1, model_version=global_version,
                                  buffered_clients=members))
            queue_history.append(dict(aggregation=global_version, virtual_time=when,
                                      pending=len(queue.heap), preview=' | '.join(queue_snapshot(queue))))
            buffer.clear()
            last_aggregation_time = when

        # Independently begin another model download after this update's arrival.
        # If this event triggered an aggregation, it gets the new version on completion.
        schedule_download(c, when)

assert arrived_updates == TARGET_UPDATES
assert not buffer, 'The final update budget must be a multiple of buffer size.'
assert len(agg_metrics)-1 == TARGET_UPDATES//AGGREGATION_BUFFER_SIZE
experiment_wall_elapsed = time.perf_counter()-experiment_wall_start
print(f'\nFINISHED | {arrived_updates} updates | {global_version} aggregations | '
      f'{virtual_time:.2f} virtual seconds | {experiment_wall_elapsed:.2f} actual execution seconds | '
      f'{arrived_updates/virtual_time:.4f} updates/virtual second')


## 6. Convergence, throughput, staleness, overall time and comparison exports

**Comparable quantities:** test accuracy/loss vs cumulative *client updates*, accuracy/loss vs elapsed *virtual time*, average/cumulative throughput (updates per virtual second), and total simulated completion time. Execution wall time is also measured but can vary with Colab hardware and is not a parallel runtime measure.

To overlay the old synchronous baseline, upload its `round_metrics.csv` into Colab before running the final cell. Its five rows represent 10, 20, 30, 40 and 50 cumulative client updates. No synchronous observations are interpolated or invented.


In [ ]:
import os

metrics_df = pd.DataFrame(agg_metrics)
updates_df = pd.DataFrame(update_records)
events_df = pd.DataFrame(event_log)
queue_df = pd.DataFrame(queue_history)

print('ASYNC AGGREGATION SUMMARY')
display(metrics_df[['aggregation', 'updates_received', 'end_time', 'accuracy', 'loss',
                    'cumulative_throughput', 'mean_staleness', 'max_staleness']].round(4))
print(f'\nFINAL accuracy: {metrics_df.iloc[-1].accuracy*100:.2f}% | final loss: {metrics_df.iloc[-1].loss:.4f}')
print(f'Total virtual time: {virtual_time:.2f}s | measured Python/PyTorch wall time: {experiment_wall_elapsed:.2f}s')
print(f'Overall throughput: {TARGET_UPDATES/virtual_time:.4f} updates / virtual second')
print(f'Mean update staleness: {updates_df.staleness.mean():.2f} versions | max: {updates_df.staleness.max()}')

baseline = None
if os.path.exists(BASELINE_METRICS_FILE):
    baseline = pd.read_csv(BASELINE_METRICS_FILE)
    if {'round', 'end_time', 'accuracy', 'loss'}.issubset(baseline.columns):
        baseline['updates_received'] = baseline['round'] * NUM_CLIENTS
        baseline['throughput'] = np.divide(baseline['updates_received'], baseline['end_time'],
                                             out=np.zeros(len(baseline), dtype=float),
                                             where=baseline['end_time'].to_numpy() > 0)
        print('\nSYNCHRONOUS BASELINE FOUND: overlaying original CSV results.')
        print(f'Baseline: {len(baseline)*NUM_CLIENTS} updates, {baseline.end_time.iloc[-1]:.2f} virtual seconds, '
              f'{baseline.accuracy.iloc[-1]*100:.2f}% accuracy, '
              f'{baseline.updates_received.iloc[-1]/baseline.end_time.iloc[-1]:.4f} updates/s')
    else:
        print('Baseline CSV columns not recognized; skipping overlay.')
        baseline = None
else:
    print('\nOptional: upload the synchronous round_metrics.csv for automatic curve overlays.')

# Plots are separate and readable; no per-quantum console spam.
for x_async, x_sync, xlabel, title in [
    ('updates_received', 'updates_received', 'Cumulative arriving client updates', 'Accuracy vs received updates'),
    ('end_time', 'end_time', 'Elapsed virtual seconds', 'Accuracy vs simulated time')]:
    plt.figure(figsize=(9,4.5))
    plt.plot(metrics_df[x_async], metrics_df.accuracy*100, marker='o', markersize=3, label='Async K=5')
    if baseline is not None:
        plt.plot(baseline[x_sync], baseline.accuracy*100, marker='s', label='Sync all 10')
    plt.xlabel(xlabel); plt.ylabel('Test accuracy (%)'); plt.title(title)
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(9,4.5))
plt.plot(metrics_df.end_time, metrics_df.loss, marker='o', markersize=3, label='Async K=5')
if baseline is not None:
    plt.plot(baseline.end_time, baseline.loss, marker='s', label='Sync all 10')
plt.xlabel('Elapsed virtual seconds'); plt.ylabel('Cross-entropy test loss')
plt.title('Convergence (loss) vs simulated time'); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(9,4.5))
plt.plot(metrics_df.end_time, metrics_df.cumulative_throughput, marker='o', label='Async K=5')
if baseline is not None:
    plt.plot(baseline.end_time, baseline.throughput, marker='s', label='Sync all 10')
plt.xlabel('Elapsed virtual seconds'); plt.ylabel('Cumulative updates / virtual second')
plt.title('Throughput vs simulated time'); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(9,4))
plt.hist(updates_df.staleness, bins=np.arange(updates_df.staleness.max()+2)-0.5, edgecolor='black')
plt.xlabel('Server versions behind at arrival'); plt.ylabel('Number of client updates')
plt.title('Observed model staleness — intentionally uncorrected'); plt.tight_layout(); plt.show()

metrics_df.to_csv(ASYNC_METRICS_FILE, index=False)
updates_df.to_csv('async_client_updates.csv', index=False)
events_df.to_csv('async_scheduler_events.csv', index=False)
queue_df.to_csv('async_priority_queue_snapshots.csv', index=False)
print('Saved async_staleness_unweighted_k5_metrics.csv, async_client_updates.csv, '
      'async_scheduler_events.csv, async_priority_queue_snapshots.csv')
